In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd

# Add project root directory to path
sys.path.append(os.path.abspath(".."))

from src.data import load_raw_dataset
from src.features import build_feature_pipeline
from src.models import train_lgb_time_series

In [2]:
# 1. Load training dataset (Set nrows=None for full training set before submitting)
df_train_raw = load_raw_dataset(data_dir="../data/raw", split="train", nrows=100000)

# 2. Build training features
df_train_fe = build_feature_pipeline(df_train_raw)

# 3. Train fold models
drop_cols = ["TransactionID", "TransactionDT", "isFraud", "uid1", "uid2"]

models, oof_preds, mean_auc = train_lgb_time_series(
    df=df_train_fe,
    target_col="isFraud",
    drop_cols=drop_cols,
    n_splits=5
)

print(f"\nModel training complete with Mean CV ROC-AUC: {mean_auc:.5f}")

Loading train_transaction.csv...
Loading train_identity.csv...
Performing left join on TransactionID...
Optimizing memory footprint...
Memory usage decreased to 176.91 MB (46.6% reduction)
Starting feature engineering pipeline...
Feature engineering complete. Total columns: 453
Starting Time-Series CV (5 Folds) on 448 features...
Fold 1 ROC-AUC: 0.89015
Fold 2 ROC-AUC: 0.91791
Fold 3 ROC-AUC: 0.91483
Fold 4 ROC-AUC: 0.92367
Fold 5 ROC-AUC: 0.90588
-----------------------------------
Mean CV ROC-AUC: 0.91049
-----------------------------------

Model training complete with Mean CV ROC-AUC: 0.91049


In [ ]:
# 1. Load full raw test set
df_test_raw = load_raw_dataset(data_dir="../data/raw", split="test", nrows=None)

# 2. Apply identical feature engineering pipeline
df_test_fe = build_feature_pipeline(df_test_raw)

# 3. Align features with training column ordering
feature_cols = [c for c in df_train_fe.columns if c not in drop_cols]
X_test = df_test_fe[feature_cols].copy()

# 4. Enforce categorical dtypes matching LightGBM requirements
cat_cols = X_test.select_dtypes(include=["object", "string"]).columns
for c in cat_cols:
    X_test[c] = X_test[c].astype("category")

print(f"Test feature matrix shape: {X_test.shape}")

Loading test_transaction.csv...


In [ ]:
# Initialize prediction array
test_preds = np.zeros(len(X_test))

# Average probability outputs across all trained fold models
print("Generating out-of-fold test predictions...")
for i, model in enumerate(models, 1):
    fold_pred = model.predict_proba(X_test)[:, 1]
    test_preds += fold_pred / len(models)
    print(f"Fold {i} predictions weighted.")

print(f"Inference complete. Mean test predicted probability: {test_preds.mean():.4f}")

In [ ]:
# Ensure output directory exists
os.makedirs("../data/processed", exist_ok=True)
sub_path = "../data/processed/submission.csv"

# Construct submission DataFrame
sub = pd.DataFrame({
    "TransactionID": df_test_raw["TransactionID"],
    "isFraud": test_preds
})

# Save to CSV
sub.to_csv(sub_path, index=False)

print("=" * 50)
print("          SUBMISSION FILE CREATED              ")
print("=" * 50)
print(f"Saved location: {sub_path}")
print(f"Total Rows:     {len(sub):,}")
print(f"Columns:        {list(sub.columns)}")
print("\nFirst 5 rows:")
print(sub.head())
print("=" * 50)